In [ ]:
# STEP 0 — 라이브러리 임포트 및 전역 설정

import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from typing import Optional, Tuple, List

## STEP 1 — 카메라 파라미터 정의

KITTI odometry sequence 09의 카메라 파라미터를 정의합니다.

**Projection Matrix** 구조:
$$P = K \cdot [R \mid t] \quad (3 \times 4)$$

- **K** : 초점거리·주점 등 내부 파라미터 (3×3)
- **R** : 카메라 회전 (3×3)
- **t** : 카메라 이동 (3×1)
- **h** : 카메라 지면으로부터 높이 [m] (KITTI ≈ 1.65 m)

> 실제 데이터 사용 시 `calib.txt`의 P0 행을 파싱하여 교체하세요.

In [ ]:
def get_kitti_camera_params() -> dict:
    """KITTI sequence 09 기준 카메라 파라미터를 반환합니다."""

    # KITTI P0 (grayscale left camera) — calib.txt에서 발췌
    P = np.array([
        [707.0912,   0.0,      601.8873, 0.0],
        [  0.0,    707.0912,  183.1104, 0.0],
        [  0.0,      0.0,      1.0,     0.0]
    ], dtype=np.float64)

    K = P[:3, :3].copy()          # 내부 파라미터
    R = np.eye(3, dtype=np.float64)   # 기준 카메라: identity
    t = np.zeros((3, 1), dtype=np.float64)
    h = 1.65                      # 카메라 높이 [m]

    return {"K": K, "P": P, "R": R, "t": t, "h": h}


# 파라미터 로드 및 확인
params = get_kitti_camera_params()
K, P, R, t, h = params["K"], params["P"], params["R"], params["t"], params["h"]

print("K (내부 파라미터):\n", K)
print("\nP (Projection Matrix):\n", P)
print(f"\n카메라 높이 h = {h} m")

In [ ]:
# STEP 2 — 소실점 추정 및 사다리꼴 ROI 생성

def estimate_vanishing_point(img_h: int, img_w: int,
                              K: np.ndarray, R: np.ndarray) -> Tuple[int, int]:
    """
    K와 R로 소실점 픽셀 좌표를 계산합니다.

    도로가 Z축(전방) 방향으로 뻗어있다고 가정:
        d_3D = [0, 0, 1]ᵀ
        vp_homog = K · R · d_3D
        vp = vp_homog[:2] / vp_homog[2]
    """
    d_forward = np.array([0.0, 0.0, 1.0])
    vp_homog  = K @ R @ d_forward
    vp_x = int(vp_homog[0] / vp_homog[2])
    vp_y = int(vp_homog[1] / vp_homog[2])
    vp_x = int(np.clip(vp_x, 0, img_w - 1))
    vp_y = int(np.clip(vp_y, 0, img_h - 1))
    return vp_x, vp_y


def create_trapezoid_roi(img_h: int, img_w: int,
                          vp_x: int, vp_y: int,
                          bottom_ratio: float = 0.08,
                          top_ratio:    float = 0.35) -> np.ndarray:
    """
    소실점을 상단 기준으로 하는 사다리꼴 ROI 마스크를 생성합니다.

    Args:
        bottom_ratio: 이미지 하단 좌우 여백 비율
        top_ratio:    소실점 근처 상단 너비 비율 (좁게)

    Returns:
        mask: uint8, 255=ROI 내부(도로 후보), 0=외부
    """
    mask  = np.zeros((img_h, img_w), dtype=np.uint8)
    y_top = int(img_h * 0.55)      # 상단 경계 (소실점 아래 여유)
    y_bot = img_h - 1

    half_top = int(img_w * top_ratio)
    half_bot = int(img_w * (0.5 - bottom_ratio))

    pts = np.array([
        [vp_x - half_top, y_top],
        [vp_x + half_top, y_top],
        [vp_x + half_bot, y_bot],
        [vp_x - half_bot, y_bot],
    ], dtype=np.int32)

    cv2.fillPoly(mask, [pts], 255)
    return mask